# Huấn luyện Mô hình Seq2Seq LSTM + Luong Attention trên Kaggle

**Lưu ý thiết lập quan trọng trước khi chạy:**
1. Ở cột bên phải (**Settings**):
   - **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100** (miễn phí 30 giờ/tuần).
   - **Internet**: Chuyển sang **ON** (bắt buộc để tải thư viện và dữ liệu).

In [ ]:
# Kiểm tra card đồ họa GPU
!nvidia-smi

## 1. Lấy mã nguồn từ GitHub về Kaggle

- Điền GitHub Personal Access Token (PAT) vào biến `GITHUB_TOKEN` (nếu repo ở chế độ Private).

In [ ]:
GITHUB_TOKEN = "ghp_xxxxxxxxxxxx"  # Thay bằng token cá nhân của bạn
!git clone -b Hiếu https://{GITHUB_TOKEN}@github.com/vha06/dich-may-Anh-Viet.git

## 2. Di chuyển vào thư mục dự án và cài đặt thư viện

In [ ]:
%cd /kaggle/working/dich-may-Anh-Viet
!pip install -q "tokenizers>=0.20.0" "sacrebleu==2.6.0" pyyaml tqdm

## 3. Tải và xử lý dữ liệu theo chuẩn Mảng 1 đồng bộ
- Giải mã HTML entities, chuẩn hóa Unicode NFC, giữ nguyên hoa/thường (`131,942` train / `1,553` val / `1,268` test).
- Huấn luyện 2 bộ BPE riêng (`bpe_en.json` và `bpe_vi.json`, mỗi bộ `8,000` tokens).

In [ ]:
!python scripts/download_data.py
!python -m src.data.preprocess
!python -m src.data.tokenizer

## 4. Huấn luyện mô hình Seq2Seq LSTM + Luong Attention (có `BucketBatchSampler` & `pack_padded_sequence`)

Mô hình có điểm SacreBLEU cao nhất trên tập `val` sẽ tự động lưu vào `checkpoints/best_seq2seq.pt`.

In [ ]:
!PYTHONPATH=. python scripts/train.py --config configs/baseline_lstm.yaml

## 5. Nạp checkpoint và Dịch thử câu thực tế (Greedy & Beam Search)

In [ ]:
import torch
from scripts.train import load_config, build_model
from src.data.tokenizer import load_tokenizer
from src.evaluation.decode import greedy_decode, beam_search_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = load_config("configs/baseline_lstm.yaml")
src_tok = load_tokenizer("data/tokenizer/bpe_en.json")
tgt_tok = load_tokenizer("data/tokenizer/bpe_vi.json")

model = build_model(
    cfg,
    src_vocab_size=src_tok.vocab_size,
    tgt_vocab_size=tgt_tok.vocab_size,
    pad_id=src_tok.pad_id,
    device=device,
)

ckpt = torch.load("checkpoints/best_seq2seq.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"✓ Đã nạp checkpoint Epoch {ckpt['epoch']} (Best BLEU = {ckpt['bleu']:.2f})")

test_sentences = [
    "How are you today ?",
    "I am learning machine translation with deep learning .",
    "Vietnam is a beautiful country in Southeast Asia .",
    "Thank you very much for your help .",
]

print("\n=== KẾT QUẢ DỊCH THỰC TẾ ===")
for s in test_sentences:
    ids = src_tok.encode(s, add_special_tokens=True, max_len=128)
    src_ids = torch.tensor([ids], dtype=torch.long, device=device)
    src_lengths = torch.tensor([len(ids)], dtype=torch.long, device=device)
    greedy_res, _ = greedy_decode(model, src_ids, tgt_tok, max_len=128, device=device, src_lengths=src_lengths)
    beam_res = beam_search_decode(model, src_ids, tgt_tok, beam_width=5, max_len=128, device=device, src_lengths=src_lengths)
    print(f"\n[EN]: {s}")
    print(f"[Greedy]: {greedy_res[0]}")
    print(f"[Beam  ]: {beam_res[0]}")

## 6. Copy checkpoint ra `/kaggle/working/` để tải về máy

In [ ]:
!cp /kaggle/working/dich-may-Anh-Viet/checkpoints/best_seq2seq.pt /kaggle/working/best_seq2seq.pt
%cd /kaggle/working
from IPython.display import FileLink
FileLink("best_seq2seq.pt")